## Output Parser
**Output Parser**는 대규모 언어 모델(LLM, Large Language Model)의 출력 결과를 **애플리케이션에서 활용할 수 있도록 적절한 형식으로 변환**하는 도구이다.
- LLM은 일반적으로 텍스트 형태로 응답을 생성하지만, 이 텍스트는 그대로 활용하기 어려운 경우가 많다.
- Output Parser는 이러한 **비구조적 텍스트 데이터를 구조화된 데이터로 변환**하여 프로그램에서 활용 가능하도록 만든다.
- 예를 들어, 키워드 리스트를 뽑거나 JSON 형식으로 정보를 변환하는 데 사용된다.

## 주요 Output Parser 종류

1. **CommaSeparatedListOutputParser**
   - 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
   - 예: `"사과, 바나나, 포도"` → `["사과", "바나나", "포도"]`
2. **JsonOutputParser**
   - LLM의 출력이 JSON 형식일 때 이를 Python의 `dict` 객체로 변환한다.
   - JSON(JavaScript Object Notation)은 데이터 구조를 표현하기 위한 경량 포맷이다.
3. **PydanticOutputParser**
   - JSON 데이터를 Python의 [Pydantic](https://docs.pydantic.dev) 모델로 변환한다. - 객체로 바꾸겠다.
   - Pydantic은 데이터 유효성 검사와 설정 관리에 널리 사용되는 Python 라이브러리이다.
4. **StrOutputParser**
   - 모델의 출력 결과를 단순 문자열로 반환한다. - .content 뽑아내기 
   - Chat 기반 모델은 Message 객체의 속성으로 LLM 결과를 반환한다. 거기에서 응답 문자열만 추출해서 반환한다.
> `JsonOutputParser`, `PydanticOutputParser` 는 모두 Pydantic을 사용해 데이터 구조(schema)를 정의하고, 해당 구조에 따라 출력을 검증하고 변환한다.
> 위처럼 정확히 정해진 형태로 출력이 나와야 파싱이 가능하므로 이를 위해 프롬프트를 올바르게 짜야한다.

## 주요 메소드
- `parse(text: str)`
  - LLM이 생성한 문자열(string) 응답을 받아 정해진 구조(list, json, instance, str)로 변환하여 반환한다.
- `get_format_instructions() -> str`
  - 각 OutputParer가 변환할 수있는 형식으로 LLM이 응답하도록 하는 프롬프트 텍스트를 반환한다.
   - -> 위 파서들에 맞는 프롬프트들을 미리 만들어놨어.
  - 이 내용을 프롬프트에 넣어서 LLM이 정확한 포맷으로 응답하도록 유도한다.
  
## 참고
- Output Parser는 일반적으로 [`Runnable`](05_chaing_LECL.ipynb#Runnable) 인터페이스를 상속하여 구현되며, `invoke()` 메서드를 통해 실행할 수 있다.
- `invoke()`는 내부적으로 `parse()`를 호출하여 동작한다.
- 필요한 경우 Output Parser를 직접 구현하여 사용자 정의 출력 포맷을 처리할 수도 있다. 


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## StrOutputParser
- 모델(LLM)의 출력 결과를 string으로 변환하여 반환하는 output parser.
- Chat Model은  Message 객체에서 content 속성값을 추출하여 문자열로 반환한다.

In [ ]:
# from langchain import OpenAI, HuggingFacePipeline, PromptTemplate

# from langchain_openai import OpenAI
# from langchain_huggingface import HuggingFacePipeline
# from langchain_core.prompts import PromptTemplate

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 프롬프트 생성
prompt_template = ChatPromptTemplate.from_template(
    "한국의 {topic} 관련된 속담을 {count}개 알려줘."
)
prompt = prompt_template.format(topic="호랑이", count=2)

# LLM 모델 생성
model = ChatOpenAI(model_name="gpt-4o-mini")

# LLM 모델에 prompt를 전달하고 응답 받기.
## prompt -> llm model -> response
response = model.invoke(prompt)

In [ ]:
prompt_template # HumanMessagePromptTemplate 객체 하나 가짐

ChatPromptTemplate(input_variables=['count', 'topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['count', 'topic'], input_types={}, partial_variables={}, template='한국의 {topic} 관련된 속담을 {count}개 알려줘.'), additional_kwargs={})])

In [15]:
prompt

'Human: 한국의 호랑이 관련된 속담을 2개 알려줘.'

In [11]:
from pprint import pprint
print('response_metadata : ')
pprint(response.response_metadata)
print('response_metadata : ')
pprint(response.usage_metadata)


response_metadata : 
{'finish_reason': 'stop',
 'id': 'chatcmpl-Bk1yKdwpt2YFLe85u17tuGUOS6LWx',
 'logprobs': None,
 'model_name': 'gpt-4o-mini-2024-07-18',
 'service_tier': 'default',
 'system_fingerprint': 'fp_34a54ae93c',
 'token_usage': {'completion_tokens': 161,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': 0,
                                               'reasoning_tokens': 0,
                                               'rejected_prediction_tokens': 0},
                 'prompt_tokens': 25,
                 'prompt_tokens_details': {'audio_tokens': 0,
                                           'cached_tokens': 0},
                 'total_tokens': 186}}
response_metadata : 
{'input_token_details': {'audio': 0, 'cache_read': 0},
 'input_tokens': 25,
 'output_token_details': {'audio': 0, 'reasoning': 0},
 'output_tokens': 161,
 'total_tokens': 186}


In [5]:
print("응답결과:", response.content)

응답결과: 한국의 호랑이와 관련된 속담 두 개는 다음과 같습니다.

1. **호랑이 굴에 가야 호랑이 새끼를 잡는다.**  
   - 이 속담은 목표를 이루기 위해서는 위험을 감수해야 한다는 의미로, 도전하지 않으면 성과를 얻을 수 없다는 것을 강조합니다.

2. **호랑이의 꼬리를 밟지 마라.**  
   - 이 속담은 강한 사람이나 세력에 함부로 가까이 가지 말라는 경고의 의미로, 위험한 상대에게 불필요하게 시비를 걸지 말라는 뜻입니다.

이 두 속담은 호랑이의 위세와 위험성을 잘 나타내고 있습니다.


In [13]:
parser = StrOutputParser()  # Message 객체에서 content 속성(메세지)의 값만 추출.
res = parser.invoke(response) # LLM 모델의 응답결과
print(res)

한국에서 호랑이와 관련된 속담은 다음과 같습니다.

1. **"호랑이 굴에 가야 호랑이 새끼를 잡는다."**
   - 의미: 원하는 것을 얻으려면 위험을 감수해야 한다는 뜻입니다.

2. **"호랑이 담배 피우던 시절."**
   - 의미: 아주 오래전의 일을 이야기할 때 사용되며, 과거를 회상하는 표현입니다.

이 속담들은 한국 문화에서 호랑이의 위엄과 관련된 의미를 담고 있습니다.


In [ ]:
# prompt_template -> model -> output parser
chain = prompt_template | model | parser
# 딕셔너리 -> prompt(str or dict) -> 출력 -> 원하는 출력 

res = chain.invoke({"topic":"사람의 정신력", "count":3})
print(res)

content='한국의 정신력과 관련된 속담은 다음과 같습니다:\n\n1. **"고생 끝에 낙이 온다"**  \n   고생이나 어려움을 겪고 나면 좋은 날이 온다는 의미로, 힘든 상황을 이겨내는 정신력을 강조합니다.\n\n2. **"산 넘어 산"**  \n   문제나 어려움이 하나 해결되면 또 다른 어려움이 나타난다는 뜻으로, 계속해서 도전하고 극복해 나가는 정신력을 나타냅니다.\n\n3. **"하늘을 향해 뻗은 손은 반드시 길어진다"**  \n   목표를 향해 꾸준히 노력하는 사람은 결국 성과를 이룰 수 있다는 뜻으로, 끈기와 인내를 강조하는 속담입니다.\n\n이 속담들은 어려운 상황에서도 포기하지 않고 견디는 정신력을 잘 표현하고 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 190, 'prompt_tokens': 24, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62a23a81ef', 'id': 'chatcmpl-BgLfm4tKUxst5EN6DNkWIyyGbqjIs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--9cbc08a3-cf62-43b3-b7d7-a5b6da7a988d-0' usage_metadata={'input_tokens': 24, 'output

In [21]:
# 출력 형식을 지정하는 프롬프트를 조회
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
# parser.get_format_instructions()  # -> str 추출 / .content 추출이라 프롬프트 없음 


## CommaSeparatedListOutputParser

- 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
  - "a,b,c" => ['a','b','c']

In [22]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
res_txt = "사과,배,귤,수박,오렌지"
print(parser.invoke(res_txt))

['사과', '배', '귤', '수박', '오렌지']


In [23]:
# 출력 형식을 지정하는 프롬프트를 조회
format_string = parser.get_format_instructions()
print(format_string)


Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from textwrap import dedent # 없애기 
prompt_template = ChatPromptTemplate.from_template( # 만들때부터 템플릿 생성 
    dedent("""
    # instruction
    {subject}의 이름 다섯개를 나열해주세요.
    
    # output indicator
    {format_instructions}
    """),
    partial_variables={"format_instructions": parser.get_format_instructions()} # 프롬프트 형식 넣어주기 
)
# partial_variables={변수명:넣을값,..} : 템플릿의 placeholder 변수에 넣을 값을 
#   PromptTemplate객체를 생성할 때 넣겠다. 변수 자리에 넣을 값이 있는데 함수나 메소드호출을 통해 
#   그 값을 가져와야 하는 경우 사용.

model = ChatOpenAI(model_name="gpt-4o-mini")

# prompt 생성
prompt = prompt_template.invoke({"subject":"동물"})
# LLM에 요청
response = model.invoke(prompt)

In [6]:
# 응답 확인
print(response.content)
# parser를 이용해 List로 변환.
res = parser.invoke(response)
print(type(res))
print(res)

호랑이, 사자, 코끼리, 고양이, 강아지
<class 'list'>
['호랑이', '사자', '코끼리', '고양이', '강아지']


In [32]:
# chain으로 구성
chain = prompt_template | model | parser

res = chain.invoke({"subject":"한국산 자동차"})
res

['소나타', '아반떼', '모닝', '싼타페', 'K5']

## JsonOutputParser

- JSON 형식의 응답을 dictionary로 반환한다.
- JSON 형식을 정하려는 경우 [Pydantic](Ref_typing_Pydantic.ipynb)을 이용해 JSON 스키마를 정의하여 JsonOutputParser 생성시 전달한다.
  - Pydantic 모델클래스를 이용해 LLM 모델이 응답할 때 json의 어떤 key에 어떤 응답을 작성할 지 Field로 정의한다.
  - Schema 지정은 필수는 아니다. 
- LLM이 JSON Schema를 따르는 형태로 응답을 하면 JsonOutputParser는 Dictionary로 변환한다.

In [26]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
res_text = """
{
    "name":"홍길동",
    "age": 20,
    "address": "서울",
    "hobby": ["독서", "게임"]
}
"""
res_dict = parser.invoke(res_text)
print(type(res_dict), res_dict)
res_dict
res_dict["name"], res_dict["address"]

<class 'dict'> {'name': '홍길동', 'age': 20, 'address': '서울', 'hobby': ['독서', '게임']}


('홍길동', '서울')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

prompt = ChatPromptTemplate(

)
model = ChatOpenAI()


'Return a JSON object.'

In [27]:
prompt_template = ChatPromptTemplate.from_template(
    "{name}에 대해서 설명해줘.\n{format_instructions}",
    partial_variables={"format_instructions":parser.get_format_instructions()}
)
model = ChatOpenAI(model_name="gpt-4o-mini")

prompt = prompt_template.invoke({"name":"제네시스"})
res = model.invoke(prompt)

In [28]:
print(res.content)

```json
{
  "brand": "제네시스 (Genesis)",
  "description": "제네시스는 현대자동차의 고급차 브랜드로, 럭셔리 차량 및 SUV 모델을 주로 생산합니다. 제네시스는 품질, 디자인, 성능에 중점을 두며, 고객에게 우수한 드라이빙 경험을 제공합니다.",
  "established": "2015년",
  "headquarters": "대한민국 현대자동차 양산 공장",
  "notable_models": [
    {
      "name": "G70",
      "type": "세단",
      "release_year": 2017
    },
    {
      "name": "G80",
      "type": "세단",
      "release_year": 2016
    },
    {
      "name": "G90",
      "type": "세단",
      "release_year": 2015
    },
    {
      "name": "GV80",
      "type": "SUV",
      "release_year": 2020
    },
    {
      "name": "GV70",
      "type": "SUV",
      "release_year": 2021
    }
  ],
  "mission": "고객의 기대를 뛰어넘는 럭셔리 경험을 제공하고, 혁신적인 기술과 디자인으로 자동차 산업의 변화를 선도하는 것."
}
```


In [ ]:
res_dict = parser.invoke(res)  # res : 모델 반환값, AIMessage 객체 | res_dict : 파서 반환값, 딕셔너리 형태 
print(type(res), type(res.content), type(res_dict))

<class 'langchain_core.messages.ai.AIMessage'> <class 'str'> <class 'dict'>


In [ ]:
# 출력 스키마를 정의
## json 형식을 설계.
from pydantic import BaseModel, Field

class ItemSchema(BaseModel):
    # JSON에 포함될 항목들을 class변수로 정의. 변수명: 타입 = Field(설명)
    name: str = Field(description="제품의 이름")
    info: str = Field(description="제품에 대한 정보")
    release_date: str = Field(description="제품이 출시된 일시. yyyy-mm-dd 형식")
    price: int = Field(description="제품의 한국 가격.")

base_parser = JsonOutputParser()
parser = JsonOutputParser(pydantic_object=ItemSchema)
print(parser.get_format_instructions())     # parser 가 계승하는 클래스의 구조가 바뀌었으니 당연히 format_instructions 도 달라져 

prompt_template = ChatPromptTemplate.from_template(
    dedent("""
    # instruction
    {name}에 대해서 설명해주세요.

    # output indicator
    {format_instructions}
    """),
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

model = ChatOpenAI(model_name="gpt-4o-mini")

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "제품의 이름", "title": "Name", "type": "string"}, "info": {"description": "제품에 대한 정보", "title": "Info", "type": "string"}, "release_date": {"description": "제품이 출시된 일시. yyyy-mm-dd 형식", "title": "Release Date", "type": "string"}, "price": {"description": "제품의 한국 가격.", "title": "Price", "type": "integer"}}, "required": ["name", "info", "release_date", "price"]}
```


In [44]:
p = prompt_template.invoke({"name":"Galaxy S24"})   # p 는 프롬프트템플릿의 반환값, ChatPromptValue 객체 
print(type(p), p, '\n', p.messages[0].content)
res = model.invoke(p)                               # res 는 모델의 반환값, json 형식의 AIMessage
print(type(res), res.content)
response = parser.invoke(res)                       # response 는 파서의 반환값, 딕셔너리 형태
print(type(response))

<class 'langchain_core.prompt_values.ChatPromptValue'> messages=[HumanMessage(content='\n# instruction\nGalaxy S24에 대해서 설명해주세요.\n\n# output indicator\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "제품의 이름", "title": "Name", "type": "string"}, "info": {"description": "제품에 대한 정보", "title": "Info", "type": "string"}, "release_date": {"description": "제품이 출시된 일시. yyyy-mm-dd 형식", "title": "Release Date", "type": "string"}, "price": {"description": "제품의 한국 가격.", "title": "Price", "type": "integer"}}, "required": ["name", "info", "release_date", "pri

In [67]:
response

{'name': 'Galaxy S24',
 'info': '삼성의 최신 스마트폰으로, 향상된 카메라 기능과 더 강력한 프로세서를 탑재하여 사용자의 다양한 요구를 충족합니다.',
 'release_date': '2024-01-20',
 'price': 1200000}

## PydanticOutputParser

- JSON 형태로 받은 응답을 Pydantic 모델로 변환하여 반환한다.
- 구현은 JsonOutputParser와 동일한데 parsing 결과를 pydantic 모델타입으로 반환한다.

In [45]:
from langchain_core.output_parsers import PydanticOutputParser
    
parser = PydanticOutputParser(pydantic_object=ItemSchema)
# JsonOutputParser와 동일한 format instruction을 생성.
## 응답: JSON ->  Parser: Pydatic Model객체로 변환.
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "제품의 이름", "title": "Name", "type": "string"}, "info": {"description": "제품에 대한 정보", "title": "Info", "type": "string"}, "release_date": {"description": "제품이 출시된 일시. yyyy-mm-dd 형식", "title": "Release Date", "type": "string"}, "price": {"description": "제품의 한국 가격.", "title": "Price", "type": "integer"}}, "required": ["name", "info", "release_date", "price"]}
```


In [46]:
prompt_template = ChatPromptTemplate.from_template(
    dedent("""
    # instruction
    {name}에 대해서 설명해주세요.

    # output indicator
    {format_instructions}
    """),
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

model = ChatOpenAI(model_name="gpt-4o-mini")

In [47]:
prompt = prompt_template.invoke({"name":"Mac Book"})
res = model.invoke(prompt)
response = parser.invoke(res)

In [50]:
response

ItemSchema(name='MacBook Pro', info='MacBook Pro는 애플이 제작한 고성능 노트북으로, 다양한 멀티미디어 작업에 적합하며, Retina 디스플레이, M1/M2 칩, 긴 배터리 수명 등을 특징으로 합니다.', release_date='2021-10-18', price=2390000)

In [48]:
print(type(response))

<class '__main__.ItemSchema'>


In [49]:
print("제품이름:", response.name)
print("정보:", response.info)
print(response.release_date, response.price)

제품이름: MacBook Pro
정보: MacBook Pro는 애플이 제작한 고성능 노트북으로, 다양한 멀티미디어 작업에 적합하며, Retina 디스플레이, M1/M2 칩, 긴 배터리 수명 등을 특징으로 합니다.
2021-10-18 2390000


In [77]:
chain = prompt_template | model | parser

response = chain.invoke({"name":"아이폰"})

In [78]:
response

ItemSchema(name='아이폰', info='아이폰은 애플(Apple)에서 개발한 스마트폰으로, 매력적인 디자인과 사용자 친화적인 인터페이스, 강력한 성능을 자랑합니다. 다양한 애플리케이션과 서비스를 제공하며, 전 세계적으로 많은 사용자층을 보유하고 있습니다.', release_date='2023-09-12', price=1250000)

In [78]:
# self-shot_prev
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 채팅 형태 템플릿 작성 - 튜플로 이루어진 리스트
template = [
	('system', '당신은 {subject} 전문가입니다.'),
    MessagesPlaceholder(variable_name='history', optional=True),
    ('human', '2문장이내로 {object}에 대해 설명하세요.')
]

# 프롬프트 템플릿 생성
prompt_template = ChatPromptTemplate(messages=template)
prompt_template

# 채팅 내역
chat_history = [
	('human', 'prompt에 대해 설명하세요.'),
    ('ai', 'prompt는 모델에 들어가는 어쩌구~')
]

# 프롬프트 작성
prompt = prompt_template.invoke(
	{'history':chat_history,
    'subject':'AI',
    'object':'template'}
)
prompt


ChatPromptValue(messages=[SystemMessage(content='당신은 AI 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='prompt에 대해 설명하세요.', additional_kwargs={}, response_metadata={}), AIMessage(content='prompt는 모델에 들어가는 어쩌구~', additional_kwargs={}, response_metadata={}), HumanMessage(content='2문장이내로 template에 대해 설명하세요.', additional_kwargs={}, response_metadata={})])

In [76]:
# self-shot_prev
from langchain_core.prompts import PromptTemplate

# 템플릿 작성
template = "###Instruction\n당신은 {subject} 전문가입니다. 2문장이내로 {object}에 대해 설명하세요. \n### Output Indicator : 반환값은 문자열이 아닌 AIMessage 객체를 이용하세요."

# 프롬프트 템플릿 생성
prompt_template = PromptTemplate(template=template)
prompt_template

# 프롬프트 작성
prompt = prompt_template.invoke({'subject':'AI', 'object':'prompt'})
prompt

StringPromptValue(text='###Instruction\n당신은 AI 전문가입니다. 2문장이내로 prompt에 대해 설명하세요. \n### Output Indicator : 반환값은 문자열이 아닌 AIMessage 객체를 이용하세요.')

In [ ]:
# self-shot
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

# 모델 객체 생성
model = ChatOpenAI()

# 파서 객체 생성
str_parser = StrOutputParser()

# Parser X
response = model.invoke(prompt)
response

# Parser O
res = parser.invoke(response)
res

'템플릿은 일정한 형식을 가진 문서나 파일의 공통된 틀이며, 재사용이 가능한 양식이다. 템플릿을 사용하면 일관된 디자인이나 구조를 갖춘 콘텐츠를 빠르게 생성할 수 있습니다.'

In [107]:
# comma~parser
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 파서 객체 생성
list_parser = CommaSeparatedListOutputParser()

# 프롬프트 추출 by 메소드 
prompt = "Output Instructions : " + list_parser.get_format_instructions()
prompt

# 최종
res = model.invoke("과일 종류 5개" + prompt)
response = list_parser.invoke(res)
response

['사과', '바나나', '딸기', '수박', '포도']

In [108]:
# jsonparser
from langchain_core.output_parsers import JsonOutputParser

# 파서 객체 생성
parser = JsonOutputParser()

# 프롬프트 추출
prompt = "Output Instructions : " + parser.get_format_instructions()

# 최종
res = model.invoke("파파야에 대해" + prompt)
response = parser.invoke(res)
response

{'name': 'Papaya',
 'scientific_name': 'Carica papaya',
 'origin': 'Tropical regions of the Americas',
 'appearance': 'Oval-shaped fruit with green skin that turns yellow when ripe, with orange or pink flesh and black seeds in the center',
 'nutritional_benefits': ['High in vitamin C and vitamin A',
  'Good source of fiber and folate',
  'Contains enzymes like papain that aid in digestion'],
 'culinary_uses': ['Eaten fresh as a fruit',
  'Used in salads, salsas, smoothies, and desserts',
  'Papaya seeds can be dried, ground, and used as a pepper substitute'],
 'fun_fact': 'Papayas are also known as pawpaws and are the fruit of the Carica papaya plant'}

In [142]:
# pydantic
# 스키마 작성
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from textwrap import dedent

class MySchema(BaseModel):
    #
    name: str = Field(description='제품의 이름')
    info: str = Field(description="제품의 정보")
    price: int = Field(description='제품의 가격', ge=0, le=320_000)

# 파서 객체 생성
json_parser = PydanticOutputParser(pydantic_object=MySchema)

# 프롬프트 작성
template = dedent("""
	### Instructions : {query}
	### Output Indicator : {format}
    """)
prompt_template = PromptTemplate(template=template, partial_variables={'format':json_parser.get_format_instructions()})
prompt = prompt_template.invoke({'query':'클라이밍 드라고 lv 신발'})

# 최종
res = model.invoke(prompt)
response = json_parser.invoke(res)

In [143]:
response

MySchema(name='Climbing shoe', info='Lowa Renegade GTX Mid', price=250000)